# Python Keywords — From Basics to Top 1% Understanding

Keywords are reserved words baked into the language's grammar. You can't use them as variable, function, or class names because the parser needs them unambiguously to understand code structure.

This notebook covers:
1. Getting the full keyword list programmatically
2. What happens when you misuse a keyword (and how to identify keywords)
3. Keywords grouped by category, with working examples for each
4. **Keywords vs. soft keywords** — a distinction the basic article misses entirely
5. **Keywords vs. identifiers vs. builtins** — three different things people conflate
6. `async`/`await` in action
7. A quiz to test yourself

## 1. Getting the List of Keywords

In [1]:
import keyword

print("The list of keywords are:")
print(keyword.kwlist)
print("\nTotal count:", len(keyword.kwlist))

The list of keywords are:
['False', 'None', 'True', 'and', 'as', 'assert', 'async', 'await', 'break', 'class', 'continue', 'def', 'del', 'elif', 'else', 'except', 'finally', 'for', 'from', 'global', 'if', 'import', 'in', 'is', 'lambda', 'nonlocal', 'not', 'or', 'pass', 'raise', 'return', 'try', 'while', 'with', 'yield']

Total count: 35


### Checking whether a specific word is a keyword

Don't memorize the list — ask Python directly.

In [2]:
print(keyword.iskeyword("for"))
print(keyword.iskeyword("total_score"))
print(keyword.iskeyword("class"))
print(keyword.iskeyword("match"))   # False -- see soft keywords below!

True
False
True
False


## 2. Identifying Keywords

- **Syntax highlighting**: most IDEs color keywords differently from your own names.
- **`SyntaxError`**: Python raises this immediately if you try to use a keyword where an identifier is expected.

In [3]:
# You can't actually run this directly in a notebook cell (it's a SyntaxError at PARSE time,
# before any code runs) -- so we demonstrate it via exec() on a string instead.
try:
    exec("for = 10\nprint(for)")
except SyntaxError as e:
    print("SyntaxError:", e)

SyntaxError: invalid syntax (<string>, line 1)


**Why this matters more than it looks:** a `SyntaxError` happens at *compile time*, before your program even starts running — unlike a `NameError` or `TypeError`, which only shows up when that specific line actually executes. This is why a stray keyword-as-variable-name breaks the *entire file*, even code that would never have reached that line.

## 3. Keywords by Category — With Working Examples

The article gives you the categories. Here's each one actually running, so you see *why* each keyword needs to be reserved.

### Value keywords: `True`, `False`, `None`

These are the only keywords that are also literal *values* — they refer to actual singleton objects.

In [4]:
print(type(True), type(False), type(None))
print(True + True)     # bool IS a subclass of int -- this really prints 2
print(None is None)    # None is a singleton; always compare with 'is', not '=='

<class 'bool'> <class 'bool'> <class 'NoneType'>
2
True


### Operator keywords: `and`, `or`, `not`, `is`, `in`

Unlike `+` or `==`, these operators are spelled as *words* rather than symbols — but they behave exactly like operators (see the earlier Operators notebook for `and`/`or` short-circuiting and `is` vs `==`).

In [5]:
x = 5
print(x > 0 and x < 10)
print(x is None or x in [1, 2, 5])
print(not False)

True
True
True


### Control flow keywords

`if / elif / else`, `for`, `while`, `break`, `continue`, `pass`, `try / except / finally`, `raise`, `assert`.

In [6]:
# if / elif / else
n = 7
if n % 2 == 0:
    print("even")
elif n % 3 == 0:
    print("divisible by 3")
else:
    print("odd, not divisible by 3")

# for / break / continue / pass
for i in range(10):
    if i == 2:
        continue    # skip this iteration
    if i == 5:
        break       # stop the loop entirely
    if i == 0:
        pass        # do nothing -- a placeholder
    print("i =", i)

odd, not divisible by 3
i = 0
i = 1
i = 3
i = 4


In [7]:
# try / except / finally / raise / assert
def divide(a, b):
    try:
        assert b != 0, "divisor cannot be zero"   # assert raises AssertionError if condition is False
        return a / b
    except ZeroDivisionError:
        print("caught a ZeroDivisionError")
        raise               # re-raise the same exception up the call stack
    except AssertionError as e:
        print("assertion failed:", e)
        return None
    finally:
        print("finally always runs, success or failure")

print(divide(10, 2))
print(divide(10, 0))

finally always runs, success or failure
5.0
assertion failed: divisor cannot be zero
finally always runs, success or failure
None


**Note on `assert`:** it's meant for catching *programmer errors* and debugging, not for validating user input in production — assertions can be globally disabled by running Python with the `-O` (optimize) flag, which strips them out entirely.

In [8]:
import sys
print("Assertions currently enabled?", __debug__)
# Running `python -O script.py` sets __debug__ to False and skips every `assert` statement in the file.

Assertions currently enabled? True


### Function and class keywords: `def`, `return`, `lambda`, `yield`, `class`

In [9]:
def square(x):
    return x * x

square_lambda = lambda x: x * x   # anonymous, single-expression function

print(square(5), square_lambda(5))

class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

p = Point(1, 2)
print(p.x, p.y)

25 25
1 2


**`yield` turns a function into a generator** — the function pauses and resumes instead of running to completion and returning once.

In [10]:
def countdown(n):
    while n > 0:
        yield n     # pauses here, returns n, resumes on next() call
        n -= 1

gen = countdown(3)
print(type(gen))
for value in gen:
    print(value)

<class 'generator'>
3
2
1


### Context management keywords: `with`, `as`

In [11]:
# 'with' guarantees cleanup (like closing a file) even if an error happens inside the block.
with open("/tmp/demo_keywords.txt", "w") as f:   # 'as' binds the context manager's return value
    f.write("hello from a context manager")

print("file closed automatically?", f.closed)   # True -- __exit__ ran even without explicit f.close()

file closed automatically? True


### Import and module keywords: `import`, `from`

In [12]:
import math
from math import sqrt as square_root   # 'as' also renames imports, not just context managers

print(math.pi)
print(square_root(16))

3.141592653589793
4.0


### Scope and namespace keywords: `global`, `nonlocal`

(Covered in depth in the Variables notebook — LEGB rule.)

In [13]:
counter = 0

def bump_global():
    global counter
    counter += 1

def outer():
    total = 0
    def inner():
        nonlocal total
        total += 1
        return total
    return inner(), inner()

bump_global()
print("counter:", counter)
print("outer():", outer())

counter: 1
outer(): (1, 2)


### Async programming keywords: `async`, `await`

In [14]:
import asyncio

async def fetch_data(name, delay):
    print(f"start fetching {name}")
    await asyncio.sleep(delay)   # pauses THIS coroutine, lets others run meanwhile
    print(f"finished fetching {name}")
    return f"{name}-result"

async def main():
    # running concurrently -- both 'sleep' periods overlap instead of stacking
    results = await asyncio.gather(
        fetch_data("A", 0.2),
        fetch_data("B", 0.1),
    )
    print("results:", results)

await main()  # Jupyter's event loop lets us 'await' directly at the top level

start fetching A
start fetching B
finished fetching B


finished fetching A
results: ['A-result', 'B-result']


Notice `B` finishes before `A` even though `A` started first — `await` yields control back to the event loop instead of blocking, so both coroutines make progress concurrently.

## 4. Keywords vs. Soft Keywords — What the Basic Article Misses

`keyword.kwlist` is **not the whole story**. Since Python 3.9, there's a second category: **soft keywords** — words that act as keywords *only in specific contexts* but are legal identifiers everywhere else. Confirmed above: `keyword.iskeyword("match")` returned `False`.

In [15]:
print("Soft keywords:", keyword.softkwlist)

Soft keywords: ['_', 'case', 'match', 'type']


- **`match` / `case`** — structural pattern matching (Python 3.10+)
- **`_`** — the wildcard pattern in `match` statements
- **`type`** — the new `type` statement for type aliases (Python 3.12+)

Proof that these are still valid variable names *outside* their special syntax:

In [16]:
match = 5        # perfectly legal! 'match' is NOT a hard keyword
type = "custom"  # also legal, though shadowing the builtin type() is bad practice
print(match, type)
del type  # restore the builtin so it doesn't stay shadowed for later cells

# But 'match' DOES have special meaning when used as a statement:
command = "start"
match command:
    case "start":
        print("Starting...")
    case "stop":
        print("Stopping...")
    case _:
        print("Unknown command")

5 custom
Starting...


**Why Python did this instead of making `match` a full keyword:** millions of existing codebases already used `match` as a variable name (e.g. `re.match`, or a local variable holding a regex match object). Making it a hard keyword would have broken that code. Soft keywords let the grammar gain new syntax without becoming backward-incompatible — a genuinely clever language design decision.

## 5. Keywords vs. Identifiers vs. Builtins — Three Different Things

People often lump these together, but they're governed by completely different rules:

| | Keyword | Builtin | Identifier |
|---|---|---|---|
| Example | `if`, `for`, `class` | `list`, `print`, `len`, `str` | `x`, `my_list`, `total` |
| Part of the *grammar*? | Yes | No | No |
| Can you use it as a variable name? | **Never** — `SyntaxError` | **Yes, but shouldn't** — silently shadows it | Yes, that's its purpose |
| What breaks if misused? | Parsing fails immediately | Nothing breaks immediately, but you lose access to the real `list`/`print`/etc. in that scope | N/A |

This distinction matters because **shadowing a builtin is legal and silent** — a much sneakier bug than misusing a keyword, which fails loudly.

In [17]:
print(keyword.iskeyword("list"))    # False -- 'list' is a builtin, NOT a keyword
print(keyword.iskeyword("print"))   # False -- 'print' is a builtin function too

# You CAN legally do this -- and it silently breaks list() for the rest of this scope:
list = [1, 2, 3]     # shadows the builtin list() constructor
try:
    new_list = list((4, 5, 6))   # this now tries to call [1,2,3](...) -- crashes!
except TypeError as e:
    print("TypeError:", e)

del list  # clean up -- restores access to the real builtin
print(list((4, 5, 6)))   # works again

False
False
TypeError: 'list' object is not callable
[4, 5, 6]


You can see every available builtin (and confirm none of them are reserved keywords) via the `builtins` module:

In [18]:
import builtins
names = [n for n in dir(builtins) if not n.startswith("_")]
print("Sample of builtins:", names[:15])
print("Total builtins:", len(names))

# Confirm: no overlap between keywords and builtins -- they're disjoint sets by design
overlap = set(keyword.kwlist) & set(names)
print("Overlap between keywords and builtins:", overlap)

Sample of builtins: ['ArithmeticError', 'AssertionError', 'AttributeError', 'BaseException', 'BaseExceptionGroup', 'BlockingIOError', 'BrokenPipeError', 'BufferError', 'BytesWarning', 'ChildProcessError', 'ConnectionAbortedError', 'ConnectionError', 'ConnectionRefusedError', 'ConnectionResetError', 'DeprecationWarning']
Total builtins: 151
Overlap between keywords and builtins: {'False', 'None', 'True'}


## 6. Quick Self-Check

In [19]:
# Q1: Which of these raise a SyntaxError if run? Predict, then test each with exec().
candidates = [
    "class = 5",
    "match = 5",
    "list = 5",
    "None = 5",
    "type = 5",
]
for code in candidates:
    try:
        exec(code)
        print(f"OK:    {code!r}")
    except SyntaxError:
        print(f"ERROR: {code!r}  -> SyntaxError")

ERROR: 'class = 5'  -> SyntaxError
OK:    'match = 5'
OK:    'list = 5'
ERROR: 'None = 5'  -> SyntaxError
OK:    'type = 5'


---
### Answer

Only `class = 5` and `None = 5` raise `SyntaxError` — both are hard keywords (`None` is even a value keyword, making it doubly forbidden). `match`, `list`, and `type` are all legal to reassign — `match` because it's a *soft* keyword, and `list`/`type` because they're merely *builtins*, not keywords at all. Reassigning them doesn't crash, it just silently shadows the originals — arguably more dangerous than a loud `SyntaxError`.

---
## Summary

| Concept | Beginner takeaway | Top 1% takeaway |
|---|---|---|
| Keywords | Reserved words you can't use as names | Reserved at the *grammar* level — misuse fails at compile time (`SyntaxError`), before the program runs |
| `keyword.kwlist` | "The list of keywords" | Only the *hard* keywords — `keyword.softkwlist` exists too |
| Soft keywords (`match`, `case`, `_`, `type`) | N/A | Special only in specific syntactic positions; legal identifiers everywhere else — a backward-compatibility trick |
| Builtins (`list`, `print`, `len`) | "Basically keywords" | Not reserved at all — you *can* shadow them, and Python won't warn you |
| `assert` | "Checks a condition" | Debug-only tool; stripped out entirely under `python -O` — never use for input validation in production |
| `yield` | "Returns a value" | Turns the function into a generator — execution pauses/resumes instead of running once |
| `async`/`await` | "Makes code async" | `await` yields control to the event loop, enabling concurrency without threads |